# 06 - Evaluation (v2)

Brings together all three candidates (Random Forest, XGBoost from `04`; MLP from `05`), compares them on the validation set, selects a winner, evaluates that winner once on the untouched held-out test set (EMBER's own official 200,000-row test split), explains a few predictions with SHAP, and saves the deployed pipeline `app.py` loads.

**Not executed in this sandbox** (needs scikit-learn/xgboost/tensorflow/shap, none installed here). Run locally: `Kernel -> Restart & Run All`.


In [11]:
# Standard library
import sys

# Third-party
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from tensorflow.keras.models import load_model

sys.path.append("../src")
from constants import ORDER_OF_FEATURES, LABEL_COLUMN, RANDOM_STATE
from explain import build_explainer, explain_prediction

pd.set_option("display.max_columns", None)

## Load and split, reload the three trained candidates

Same split logic as `02`/`04`/`05` (fixed `RANDOM_STATE`, EMBER's own `train`/`test` boundary, training-pool downsample, then an 85/15 stratified split of `train` for validation), reproduced inline here too, plus the untouched `test` rows this notebook is the only one allowed to look at.

**Compute-constrained downsample of the training pool.** `GridSearchCV`'s exhaustive search over the full ~600,000-row training pool is too slow on typical laptop hardware (each grid search trains many separate models across cross-validation folds). Downsamples `train_pool` to 75,000 rows per class (150,000 total) before the 85/15 train/validation split, using a fixed `RANDOM_STATE` so the reduction is reproducible. **The held-out `test` set (EMBER's own official 200,000-row split) is NOT reduced**: `06_evaluation.ipynb`'s final reported numbers still come from the complete, standard benchmark split, only the tuning step works on a smaller pool.


In [12]:
df = pd.read_csv("../data/dataset_pe_v2_full.csv")
train_pool = df[df["OriginalSplit"] == "train"].reset_index(drop=True)
test_df = df[df["OriginalSplit"] == "test"].reset_index(drop=True)

TRAIN_POOL_SAMPLE_PER_CLASS = 75000
train_pool = pd.concat([
    g.sample(n=min(TRAIN_POOL_SAMPLE_PER_CLASS, len(g)), random_state=RANDOM_STATE)
    for _, g in train_pool.groupby(LABEL_COLUMN)
]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print("downsampled train_pool:", train_pool.shape)

train_df = pd.concat([
    g.sample(frac=0.85, random_state=RANDOM_STATE)
    for _, g in train_pool.groupby(LABEL_COLUMN)
]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
val_df = train_pool.drop(
    pd.concat([g.sample(frac=0.85, random_state=RANDOM_STATE) for _, g in train_pool.groupby(LABEL_COLUMN)]).index
).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

X_val, y_val = val_df[ORDER_OF_FEATURES], val_df[LABEL_COLUMN]
X_test, y_test = test_df[ORDER_OF_FEATURES], test_df[LABEL_COLUMN]

rf_best = joblib.load("../models/rf_v2.joblib")
xgb_best = joblib.load("../models/xgb_v2.joblib")
mlp_scaler = joblib.load("../models/mlp_v2_scaler.joblib")
mlp_model = load_model("../models/mlp_v2.keras")
print("all three candidates loaded, val:", X_val.shape, "test:", X_test.shape)


downsampled train_pool: (150000, 23)
all three candidates loaded, val: (22500, 20) test: (200000, 20)


## Evaluate all three candidates on validation


In [13]:
print("--- Random Forest ---")
rf_val_preds = rf_best.predict(X_val)
print(classification_report(y_val, rf_val_preds, target_names=["benign", "malicious"]))

--- Random Forest ---
              precision    recall  f1-score   support

      benign       0.94      0.97      0.95     11250
   malicious       0.97      0.94      0.95     11250

    accuracy                           0.95     22500
   macro avg       0.95      0.95      0.95     22500
weighted avg       0.95      0.95      0.95     22500



In [14]:
print("--- XGBoost ---")
xgb_val_preds = xgb_best.predict(X_val)
print(classification_report(y_val, xgb_val_preds, target_names=["benign", "malicious"]))

--- XGBoost ---
              precision    recall  f1-score   support

      benign       0.94      0.95      0.95     11250
   malicious       0.95      0.94      0.95     11250

    accuracy                           0.95     22500
   macro avg       0.95      0.95      0.95     22500
weighted avg       0.95      0.95      0.95     22500



In [15]:
print("--- MLP ---")
X_val_scaled = mlp_scaler.transform(X_val)
mlp_val_proba = mlp_model.predict(X_val_scaled, verbose=0).ravel()
mlp_val_preds = (mlp_val_proba >= 0.5).astype(int)
print(classification_report(y_val, mlp_val_preds, target_names=["benign", "malicious"]))

--- MLP ---
              precision    recall  f1-score   support

      benign       0.88      0.88      0.88     11250
   malicious       0.88      0.88      0.88     11250

    accuracy                           0.88     22500
   macro avg       0.88      0.88      0.88     22500
weighted avg       0.88      0.88      0.88     22500



## Compare candidates and select a winner

Computed live from the cells above every time this notebook runs, rather than a hardcoded table, so it can never silently go stale on a rerun.

In [16]:
comparison = pd.DataFrame([
    {"Candidate": "Random Forest", "Accuracy": accuracy_score(y_val, rf_val_preds),
     "Macro F1": f1_score(y_val, rf_val_preds, average="macro"),
     "ROC-AUC": roc_auc_score(y_val, rf_best.predict_proba(X_val)[:, 1])},
    {"Candidate": "XGBoost", "Accuracy": accuracy_score(y_val, xgb_val_preds),
     "Macro F1": f1_score(y_val, xgb_val_preds, average="macro"),
     "ROC-AUC": roc_auc_score(y_val, xgb_best.predict_proba(X_val)[:, 1])},
    {"Candidate": "MLP", "Accuracy": accuracy_score(y_val, mlp_val_preds),
     "Macro F1": f1_score(y_val, mlp_val_preds, average="macro"),
     "ROC-AUC": roc_auc_score(y_val, mlp_val_proba)},
])
print(comparison.to_string(index=False))


    Candidate  Accuracy  Macro F1  ROC-AUC
Random Forest  0.951378  0.951366 0.990158
      XGBoost  0.946978  0.946975 0.988688
          MLP  0.884444  0.884444 0.956094


Set `chosen_name` below to whichever candidate has the best combination of accuracy, macro F1, and ROC-AUC in the printed comparison table above.


In [17]:
# Example (edit after seeing the real comparison table above):
chosen = xgb_best
chosen_name = "XGBoost"
is_mlp = False  # set True instead if MLP wins, evaluation code below branches on this

## Evaluate the winner on the held-out test set

This is EMBER's own official 200,000-row test split, never touched in `02`, `04`, `05`, or above. This is the one honest, final check, and (since it's EMBER's standard split) is also the number directly comparable to the published EMBER baseline (ROC AUC 0.999, arXiv:1804.04637).

In [18]:
if is_mlp:
    X_test_scaled = mlp_scaler.transform(X_test)
    test_proba = mlp_model.predict(X_test_scaled, verbose=0).ravel()
    test_preds = (test_proba >= 0.5).astype(int)
else:
    test_preds = chosen.predict(X_test)
    test_proba = chosen.predict_proba(X_test)[:, 1]

print(f"--- {chosen_name} (test, final check, run once) ---")
print(classification_report(y_test, test_preds, target_names=["benign", "malicious"]))
print("ROC-AUC:", roc_auc_score(y_test, test_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, test_preds))

--- XGBoost (test, final check, run once) ---
              precision    recall  f1-score   support

      benign       0.93      0.93      0.93    100000
   malicious       0.93      0.93      0.93    100000

    accuracy                           0.93    200000
   macro avg       0.93      0.93      0.93    200000
weighted avg       0.93      0.93      0.93    200000

ROC-AUC: 0.9792818687
Confusion matrix:
 [[92983  7017]
 [ 7232 92768]]


## Explain predictions with SHAP

SHAP explains a prediction after the model already made it: each feature value pulls the verdict toward malicious or toward benign. Global feature importance uses a random 3,000-row subsample of the test set, purely because SHAP summary plots don't get more informative past a certain sample size and full-scale plotting is wasted compute (same reasoning already noted in `02_data_preparation.ipynb`'s dataset-size discussion, this is not a re-reduction of training data, it happens after training). Skipped entirely if the MLP won, `explain.py`'s `TreeExplainer` only supports tree-based models.

In [19]:
if not is_mlp:
    explainer = build_explainer(chosen)
    shap_sample = X_test.sample(n=3000, random_state=42)
    for i in range(3):
        row = X_test.iloc[[i]]
        toward_malicious, toward_benign = explain_prediction(explainer, chosen, row, ORDER_OF_FEATURES)
        print(f"Row {i} (true label={int(y_test.iloc[i])}, predicted={int(test_preds[i])})")
        print("  toward malicious:", toward_malicious[:3])
        print("  toward benign:", toward_benign[:3])
else:
    print("MLP was selected; SHAP TreeExplainer does not apply, skipped.")

Row 0 (true label=1, predicted=1)
  toward malicious: [('SectionMaxEntropy', np.float32(1.6644243)), ('SuspiciousNameSection', np.float32(1.2081722)), ('AvgStringLength', np.float32(0.9225147))]
  toward benign: [('SectionsLength', np.float32(-0.111049674)), ('NumRegistryKeys', np.float32(-0.008355321)), ('SectionMaxVirtualsize', np.float32(0.060782004))]
Row 1 (true label=0, predicted=0)
  toward malicious: [('SuspiciousNameSection', np.float32(0.5208959)), ('NumStrings', np.float32(0.268813)), ('DirectoryEntryImport', np.float32(0.24778683))]
  toward benign: [('FileSize', np.float32(-0.6364605)), ('SectionMaxEntropy', np.float32(-0.63473266)), ('SectionsLength', np.float32(-0.58407295))]
Row 2 (true label=1, predicted=1)
  toward malicious: [('SuspiciousImportFunctions', np.float32(0.87694365)), ('StringEntropy', np.float32(0.8505479)), ('NumMZStrings', np.float32(0.6338454))]
  toward benign: [('SuspiciousNameSection', np.float32(-0.2579964)), ('SectionMaxRawsize', np.float32(-0.09

## Real-world validation

Held-out test accuracy above measures performance on EMBER's curated benchmark, not real-world reliability. `07_real_world_validation.ipynb` is the dedicated notebook for that: it scans real `.exe`/`.dll` files on disk (e.g. under `C:\Windows\System32`) and reports a genuine false-positive rate. It is kept separate because it needs real local files that are not present in every environment.


## Save the deployed pipeline

**Deployment constraint:** `app.py` only knows how to load `models/classical_pipeline.joblib` (a scikit-learn-style Pipeline with `predict_proba`). If the MLP wins and `is_mlp = True` above, the cell below saves it under a different filename (`deployed_mlp_v2.keras` + scaler) that `app.py` does not look for; the app would keep showing "No trained model found" until `app.py` is extended to also load and serve a Keras model (different loading code, manual scaling, no `predict_proba`). A classical candidate is expected to win, consistent with published findings on this data (EMBER's own baseline is a gradient-boosted tree model, not a neural network); if the MLP genuinely wins by a meaningful margin, that is worth discussing in the report as a deliberate scope decision.


In [20]:
if is_mlp:
    mlp_model.save("../models/deployed_mlp_v2.keras")
    joblib.dump(mlp_scaler, "../models/deployed_mlp_v2_scaler.joblib")
    print("Saved MLP pipeline to ../models/deployed_mlp_v2.keras + scaler")
else:
    joblib.dump({"pipeline": chosen, "feature_order": ORDER_OF_FEATURES}, "../models/classical_pipeline.joblib")
    print(f"Saved {chosen_name} pipeline to ../models/classical_pipeline.joblib")

Saved XGBoost pipeline to ../models/classical_pipeline.joblib


## Summary

Populated after a local run: final comparison table results, which candidate won and why, held-out test set metrics for the winner, SHAP sanity-check findings, and the real-world validation false-positive rate from `07` once available.
